# 指數平滑（Exponential Smoothing，1950s）

- **核心原理**：近期觀測權重較高
- **變體**：簡單指數平滑（SES）→ Holt（線性趨勢）→ Holt-Winters（加法／乘法季節）；阻尼趨勢（damped trend）；ETS 狀態空間模型
- **推薦資料集**：AirPassengers（1949–1960 年每月國際航空乘客數，單位千人，144 個月）；最後 24 個月當測試集
- **文獻**：Brown (1959)、Holt (1957)、Winters (1960)；Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*（fpp3）第 8 章

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

**第六部分的第一章**。跟前面所有章節最大的不同：

| | 前面的章節 | 時間序列 |
| --- | --- | --- |
| 樣本 | 彼此獨立，可以隨便洗牌 | **有順序**：今天跟昨天有關 |
| 切分 | 隨機切 train／test、k-fold CV | **只能用過去預測未來**：test 一定在最後；CV 用「滾動起點」（rolling origin） |
| 目標 | 給 $x$ 預測 $y$ | 給**過去的 $y$** 預測**未來的 $y$**（往後 $h$ 步） |
| 基準 | Dummy（猜最多的類別、平均值） | **naive**（明天 = 今天）、**季節性 naive**（下個月 = 去年同月） |
| 指標 | accuracy、RMSE… | MAE、RMSE、sMAPE、**MASE**（跟季節性 naive 比） |

一句話：**指數平滑 = 加權平均，越近的觀測權重越大，權重往過去以 $(1 - \alpha)^k$ 指數遞減；再把「水準」「趨勢」「季節」各自用同樣的方式平滑。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：`load_air_passengers()` 回傳一條月資料的 `pd.Series`（144 個值）；`split_series(y, test_size=24)` 切成前 120 個月（1949-01～1958-12）的 `y_train` 與最後 24 個月（1959-01～1960-12）的 `y_test`。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $y_t$ | 第 $t$ 期的觀測值（$t = 1, \dots, n$） | 乘客數（千人） |
| $n$、$m$ | 訓練序列長度、季節週期 | 120、12 |
| $h$ | 預測步數（往後幾期） | 1～24 |
| $\hat y_{t+h\mid t}$ | 用到第 $t$ 期為止的資料，對第 $t + h$ 期的預測 | |
| $\ell_t$、$b_t$、$s_t$ | 水準（level）、趨勢（trend，每期增加多少）、季節（seasonal）—— 第 $t$ 期結束時的估計 | |
| $\alpha$、$\beta$、$\gamma$ | 水準、趨勢、季節的平滑參數（0～1） | |
| $\phi$ | 阻尼係數（0～1；1 = 不阻尼） | |
| $e_t$ | 一步預測誤差 $y_t - \hat y_{t\mid t-1}$ | |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第一部分第 2 章 | 線性回歸 | 最小平方：最小化誤差平方和（SSE）；本章的參數也是最小化**一步預測誤差**的 SSE，只是沒有閉式解，要用數值最佳化。 |
| 第一部分調參流程 | README 調參流程 | 調參只用訓練資料；validation curve；one-standard-error rule（差距小於一個標準誤就選簡單的）。 |
| 第三部分第 2～4 課 | RNN／LSTM／GRU | 隱藏狀態 $h_t$ 一步一步更新、遺忘閘門決定「留多少舊的」；指數平滑的 $\ell_t = \alpha y_t + (1 - \alpha)\ell_{t-1}$ 就是一個**固定遺忘率**、線性的「一維 RNN」。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 加權平均**：$\sum_k w_k y_k$，$\sum_k w_k = 1$。例：$w = (0.5, 0.25, 0.25)$、$y = (40, 30, 20)$ → 32.5。

**2. 等比級數**：$\sum_{k=0}^{\infty}\alpha(1 - \alpha)^k = \alpha \cdot \frac{1}{1 - (1 - \alpha)} = 1$。所以 SES 展開後的權重 $\alpha, \alpha(1-\alpha), \alpha(1-\alpha)^2, \dots$ 加起來是 1（加上初始值的權重 $(1-\alpha)^t$）。例：$\alpha = 0.5$ → 0.5、0.25、0.125…；$\alpha = 0.1$ → 0.1、0.09、0.081…（衰減很慢）。

**3. 遞迴 vs 展開**：$\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}$ 展開 → $\ell_t = \sum_{k=0}^{t-1}\alpha(1-\alpha)^k y_{t-k} + (1-\alpha)^t \ell_0$。程式用遞迴（每一步 $O(1)$）。

**4. 加法 vs 乘法季節**：加法 $y = \text{水準} + \text{季節}$（季節的「幅度」固定，例如「7 月多 50 人」）；乘法 $y = \text{水準} \times \text{季節}$（季節的「比例」固定，例如「7 月多 20%」）。水準變大、季節波動也跟著變大 → 乘法。取 log 之後乘法變加法：$\log y = \log\text{水準} + \log\text{季節}$。

**5. 數值最佳化**（`scipy.optimize.minimize`）：給一個函數 $f(\theta)$ 與初始值、範圍（`bounds`），回傳讓 $f$ 最小的 $\theta$。`method="L-BFGS-B"` 支援範圍限制（例如 $0 \le \alpha \le 1$）。

#### 程式

**6. pandas 的時間索引**：`PeriodIndex`（「1949-01 這個月」）vs `DatetimeIndex`（「1949-01-01 這個時間點」）；`.to_timestamp()` 轉換。`pd.date_range(start, periods=h, freq="MS")` 產生未來的月份。
**7. `y.shift(k)`**：整條往後移 $k$ 期（前 $k$ 個變 NaN）；季節性 naive 就是 `y.shift(12)`。
**8. 負的索引**：`arr[-m:]` = 最後 $m$ 個；`arr[n + (h - 1) % m]` 會在長度 $m$ 的季節表裡循環。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。時間序列的 EDA 重點：**趨勢、季節、變異是否隨水準變大、異常值**。

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_air_passengers, split_series

y = load_air_passengers()
if isinstance(y.index, pd.PeriodIndex):
    y.index = y.index.to_timestamp()  # 統一用 DatetimeIndex（月初）
y = y.astype(float).rename("passengers")
y_train, y_test = split_series(y, test_size=24)
M = 12  # 季節週期（月資料）
print("全部：", len(y), y.index[0].date(), "～", y.index[-1].date(), "；train：", len(y_train), "；test：", len(y_test), y_test.index[0].date(), "～", y_test.index[-1].date())

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
y_train.plot(ax=axes[0], label="train")
y_test.plot(ax=axes[0], label="test")
axes[0].legend()
axes[0].set_title("AirPassengers (thousands)")
np.log(y).plot(ax=axes[1], color="tab:green")
axes[1].set_title("log(passengers)")
plt.tight_layout()
plt.show()

In [ ]:
# 季節性與變異
# 觀察：
#   - 每年的平均（水準）與每年的標準差（季節波動的大小）一起變大嗎？比例（std / mean）呢？→ 加法還是乘法季節？
#   - 哪幾個月最高、最低？每年的形狀一樣嗎？
yearly = y_train.groupby(y_train.index.year).agg(["mean", "std"])
yearly["std / mean"] = yearly["std"] / yearly["mean"]
yearly["std of log"] = np.log(y_train).groupby(y_train.index.year).std()
display(yearly.round(3).T)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for year, g in y_train.groupby(y_train.index.year):
    axes[0].plot(g.index.month, g.to_numpy(), label=str(year), color=plt.cm.viridis((year - 1949) / 10))
axes[0].set_xlabel("month")
axes[0].set_title("each year (raw)")
ratio = y_train / y_train.groupby(y_train.index.year).transform("mean")
sns.boxplot(x=ratio.index.month, y=ratio.to_numpy(), ax=axes[1])
axes[1].set_title("value / yearly mean (seasonal ratio)")
axes[1].set_xlabel("month")
plt.show()

In [ ]:
# 自我相關（autocorrelation）：y_t 跟 y_{t−k} 的相關係數
# 觀察：
#   - lag 1、12、24 的自我相關各是多少？為什麼 lag 12 特別高？
#   - 先去掉趨勢（差分 y_t − y_{t−1}）之後，lag 12 還高嗎？
def acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """樣本自我相關 r_k = Σ (x_t − x̄)(x_{t−k} − x̄) / Σ (x_t − x̄)²，k = 0..max_lag。"""
    x = np.asarray(x, dtype=float) - np.mean(x)
    denom = (x**2).sum()
    return np.array([(x[k:] * x[: len(x) - k]).sum() / denom for k in range(max_lag + 1)])


fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
for ax, (name, series) in zip(axes, [("raw", y_train.to_numpy()), ("first difference", np.diff(y_train.to_numpy()))]):
    r = acf(series, 36)
    ax.bar(np.arange(37), r)
    ax.axhline(2 / np.sqrt(len(series)), color="gray", ls=":")
    ax.axhline(-2 / np.sqrt(len(series)), color="gray", ls=":")
    ax.set_title(f"ACF: {name}")
    ax.set_xlabel("lag (months)")
plt.show()
print("raw：lag 1、12、24 =", acf(y_train.to_numpy(), 24)[[1, 12, 24]].round(3), "；差分後：", acf(np.diff(y_train.to_numpy()), 24)[[1, 12, 24]].round(3))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 趨勢：1949 → 1958 每年的平均大約變成幾倍？每年大約增加多少（千人）？是直線成長還是越來越快？
# 答：
#
# (2) 季節：
#     (a) 哪些月份最高、最低？季節的形狀每年都差不多嗎？
#     (b) 每年的 std 跟著 mean 變大嗎？std / mean 呢？取 log 之後的 std 呢？→ 你會用加法還是乘法季節？（前置複習 4）
# 答：
#
# (3) 自我相關：原始序列的 ACF 為什麼每個 lag 都很高？（提示：趨勢 —— 前後兩期都比平均高或都比平均低）差分之後，哪一個 lag 最突出？
# 答：
#
# (4) 為什麼 test 一定要放在最後 24 個月，不能隨機抽 24 個月？如果隨機抽，模型會「偷看」到什麼？
# 答：

---
## 2. Baseline

| Baseline | 預測 $\hat y_{n+h\mid n}$ | 適合 |
| --- | --- | --- |
| mean | 訓練資料的平均 | 沒有趨勢、沒有季節 |
| naive | $y_n$（最後一個值） | 隨機漫步 |
| drift | $y_n + h \cdot \frac{y_n - y_1}{n - 1}$（最後一個值 + 平均斜率） | 有趨勢 |
| **季節性 naive** | $y_{n+h-m\lceil h/m\rceil}$（去年同月） | 有季節 —— **本章主要的對手** |

**指標**（`learning_map.metrics.forecast_metrics`）：

| 指標 | 定義 | 說明 |
| --- | --- | --- |
| MAE、RMSE | $\frac{1}{H}\sum\lvert e\rvert$、$\sqrt{\frac{1}{H}\sum e^2}$ | 單位跟資料一樣（千人） |
| sMAPE | $\frac{100}{H}\sum\frac{2\lvert e\rvert}{\lvert y\rvert + \lvert\hat y\rvert}$ | 百分比 |
| **MASE** | MAE ÷（**訓練資料上**季節性 naive 的一步 MAE） | < 1 = 比「去年同月」還準 |

這一節直接在 **test** 上算 —— baseline 沒有要調的參數，看一次不會影響後面的選擇（跟分類章節的 Dummy 一樣）。

In [ ]:
from learning_map.metrics import forecast_metrics


def future_index(series: pd.Series, h: int) -> pd.DatetimeIndex:
    return pd.date_range(series.index[-1] + pd.offsets.MonthBegin(1), periods=h, freq="MS")


def baseline_forecast(series: pd.Series, h: int, kind: str, m: int = M) -> pd.Series:
    """mean／naive／drift／snaive 的 h 步預測。"""
    x = series.to_numpy()
    n = len(x)
    steps = np.arange(1, h + 1)
    if kind == "mean":
        f = np.full(h, x.mean())
    elif kind == "naive":
        f = np.full(h, x[-1])
    elif kind == "drift":
        f = x[-1] + steps * (x[-1] - x[0]) / (n - 1)
    elif kind == "snaive":
        f = x[n - m + (steps - 1) % m]
    else:
        raise ValueError(kind)
    return pd.Series(f, index=future_index(series, h))


H = len(y_test)
base_fc = {k: baseline_forecast(y_train, H, k) for k in ["mean", "naive", "drift", "snaive"]}
test_table = pd.DataFrame({k: forecast_metrics(y_test, f, y_train, season=M) for k, f in base_fc.items()}).T
display(test_table)
fig, ax = plt.subplots(figsize=(14, 4))
y.iloc[-60:].plot(ax=ax, color="k", label="actual")
for k, f in base_fc.items():
    f.plot(ax=ax, label=k, ls="--")
ax.legend()
plt.show()

In [ ]:
# TODO 2（問答）Baseline
#
# (1) 哪一個 baseline 最好？季節性 naive 的 MASE 是多少？它為什麼 > 1（提示：MASE 的分母是「訓練資料上」季節性 naive 的一步誤差；test 是往後 1～24 步）？
# (2) 季節性 naive 在 test 上系統性地偏低還是偏高？為什麼？（提示：去年同月 vs 今年 —— 趨勢）
# (3) 你預期 Holt-Winters（水準 + 趨勢 + 季節）能把季節性 naive 的哪一個缺點補起來？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 簡單指數平滑（SES）：只有水準 | 3 |
| 3B | Holt：水準 + 趨勢（可阻尼） | 4 |
| 3C | Holt-Winters：水準 + 趨勢 + 季節（加法／乘法）；多步預測 | 5、6 |
| 3D | 估計參數（最小化一步誤差的 SSE）、對照 statsmodels | 7 |
| 3E | 實驗：$\alpha$ 的意義、加法 vs 乘法 vs log、阻尼趨勢 | 8 |

三個模型都寫在同一個 class `HoltWintersScratch` 裡：`trend=None, seasonal=None` 就是 SES，`trend="add"` 是 Holt，再加 `seasonal="add"／"mul"` 是 Holt-Winters。

### 3A. 簡單指數平滑（SES）

$$\hat y_{t\mid t-1} = \ell_{t-1}, \qquad \ell_t = \alpha y_t + (1 - \alpha)\,\ell_{t-1}$$

讀法：新的水準 = 「這一期的觀測」與「上一期的水準」的加權平均。展開（前置複習 3）就是權重 $\alpha(1-\alpha)^k$ 的加權平均。預測未來每一期都是同一個值：$\hat y_{n+h\mid n} = \ell_n$（平的）。

#### 符號表（3A–3C 共用）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| $y_t$ | 這一期的觀測 | `yt` |
| $\ell_{t-1}$、$b_{t-1}$ | 上一期結束時的水準、趨勢 | `l`、`b` |
| $s_{t-m}$ | 「同一季」上一次的季節值 | `s[t]`（`s` 是一條清單：前 $m$ 個是初始季節，之後每期 append 一個） |
| $\ell_t$、$b_t$、$s_t$ | 這一期更新後的值 | `l_new`、`b_new`、`s_new` |
| $\phi$ | 阻尼（沒有趨勢時 $b = 0$，$\phi$ 不影響） | `phi` |
| deseason | 去掉季節的觀測：加法 $y_t - s_{t-m}$、乘法 $y_t / s_{t-m}$、沒有季節 $y_t$ | `deseason` |

**下標公式**：
- TODO 3：$\ell_t = \alpha\cdot\text{deseason} + (1 - \alpha)(\ell_{t-1} + \phi b_{t-1})$（SES 時 $b = 0$、deseason $= y_t$，就是上面的式子）。

#### 最小例子：TODO 3（SES）

$y = (10, 20, 30, 40)$、$\alpha = 0.5$、$\ell_0 = 10$。

1. **正確答案**（手算）：$\ell_1 = 0.5 \cdot 10 + 0.5 \cdot 10 = 10$、$\ell_2 = 15$、$\ell_3 = 22.5$、$\ell_4 = 31.25$；一步預測 $\hat y_{t\mid t-1} = \ell_{t-1}$ = $(10, 10, 15, 22.5)$；未來每一期都預測 31.25。
2. **積木**：展開式的權重 $\alpha(1-\alpha)^k$。
3. **留給你**：寫進 `_filter` 的迴圈（TODO 3）。

In [ ]:
# 最小例子：TODO 3（SES）
y_mini = np.array([10.0, 20.0, 30.0, 40.0])
# ---- 1. 正確答案：迴圈 ----
l_mini, fitted_mini = 10.0, []
for yt in y_mini:
    fitted_mini.append(l_mini)
    l_mini = 0.5 * yt + 0.5 * l_mini
print("一步預測：", fitted_mini, "；最後的水準（未來的預測）：", l_mini)

# ---- 2. 積木：展開式 ----
w = 0.5 * 0.5 ** np.arange(4)  # 給 y_4, y_3, y_2, y_1 的權重
print("展開式：Σ α(1−α)^k y_{4−k} + (1−α)^4 ℓ_0 =", (w * y_mini[::-1]).sum() + 0.5**4 * 10)

# ---- 3. 留給你 ----
# 寫完 TODO 3、執行過下一格之後，取消下面的註解來對照（應為 [10. 10. 15. 22.5] 與 [31.25 31.25]）：
# ses_mini = HoltWintersScratch().fit(pd.Series(y_mini), params=(0.5,), init=(10.0, 0.0, None)); print(ses_mini.fittedvalues_, ses_mini.forecast(2))

### 3B. Holt：水準 + 趨勢（可阻尼）

$$\hat y_{t\mid t-1} = \ell_{t-1} + \phi b_{t-1}, \qquad \ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1} + \phi b_{t-1}), \qquad b_t = \beta(\ell_t - \ell_{t-1}) + (1 - \beta)\,\phi b_{t-1}$$

- 趨勢 $b_t$ 也是加權平均：「這一期水準實際增加了多少」$(\ell_t - \ell_{t-1})$ 與「上一期的趨勢」的平均。
- **阻尼** $\phi < 1$：預測 $\hat y_{n+h\mid n} = \ell_n + (\phi + \phi^2 + \cdots + \phi^h)\,b_n$ —— 趨勢越往後越平（Gardner & McKenzie 1985；實務上常比直線趨勢準）。$\phi = 1$ 就是一般的 Holt（直線）。

**下標公式**：
- TODO 4：$b_t = \beta(\ell_t - \ell_{t-1}) + (1 - \beta)\phi b_{t-1}$ → `beta * (l_new - l) + (1 - beta) * phi * b`。

#### 最小例子：TODO 4（Holt，$\phi = 1$）

$y = (10, 20, 30, 40)$、$\alpha = \beta = 0.5$、$\ell_0 = 0$、$b_0 = 10$（一開始就猜「每期 +10」）。
**正確答案**：$\hat y_{1\mid 0} = 10$；$\ell_1 = 0.5 \cdot 10 + 0.5 \cdot 10 = 10$、$b_1 = 0.5(10 - 0) + 0.5 \cdot 10 = 10$ → 之後每一步都剛好預測對：一步預測 $(10, 20, 30, 40)$，最後 $\ell_4 = 40$、$b_4 = 10$，未來預測 $(50, 60)$。

In [ ]:
# 最小例子：TODO 4（Holt）—— 1. 正確答案：迴圈
l_h, b_h, fit_h = 0.0, 10.0, []
for yt in y_mini:
    fit_h.append(l_h + b_h)
    l_new = 0.5 * yt + 0.5 * (l_h + b_h)
    b_h = 0.5 * (l_new - l_h) + 0.5 * b_h
    l_h = l_new
print("一步預測：", fit_h, "；ℓ_4 =", l_h, "、b_4 =", b_h, "；未來：", [l_h + k * b_h for k in (1, 2)])
# 寫完 TODO 4、執行過下下一格之後，取消下面的註解來對照（應為 [10. 20. 30. 40.] 與 [50. 60.]）：
# holt_mini = HoltWintersScratch(trend="add").fit(pd.Series(y_mini), params=(0.5, 0.5), init=(0.0, 10.0, None)); print(holt_mini.fittedvalues_, holt_mini.forecast(2))

### 3C. Holt-Winters：加上季節

| | 加法季節 | 乘法季節 |
| --- | --- | --- |
| 一步預測 | $\hat y_{t\mid t-1} = \ell_{t-1} + \phi b_{t-1} + s_{t-m}$ | $(\ell_{t-1} + \phi b_{t-1})\,s_{t-m}$ |
| 水準 | $\ell_t = \alpha(y_t - s_{t-m}) + (1-\alpha)(\ell_{t-1} + \phi b_{t-1})$ | $\alpha\,y_t / s_{t-m} + (1-\alpha)(\cdots)$ |
| 趨勢 | 同 Holt | 同 Holt |
| 季節 | $s_t = \gamma(y_t - \ell_{t-1} - \phi b_{t-1}) + (1-\gamma)s_{t-m}$ | $s_t = \gamma\,y_t / (\ell_{t-1} + \phi b_{t-1}) + (1-\gamma)s_{t-m}$ |
| $h$ 步預測 | $\ell_n + \Phi_h b_n + s_{n+h-m(k+1)}$ | $(\ell_n + \Phi_h b_n)\,s_{n+h-m(k+1)}$ |

其中 $\Phi_h = \phi + \cdots + \phi^h$、$k = \lfloor (h-1)/m \rfloor$（$s_{n+h-m(k+1)}$ = 最後一年裡「同一個月」的季節值）。

季節的更新也是加權平均：「這一期觀測到的季節效果」（觀測 − 預期的水準，或觀測 ÷ 預期的水準）與「同一季上一次的季節值」。（這是 statsmodels `ExponentialSmoothing` 用的版本；有些教科書用 $\ell_t$ 而不是 $\ell_{t-1} + \phi b_{t-1}$，數值會略有不同。）

**初始值**（`_init_states`，已寫好）：$\ell_0$ = 第一年的平均；$b_0$ = （第二年平均 − 第一年平均）/ $m$；$s$ 的前 $m$ 個 = 第一年每個月 − $\ell_0$（加法）或 ÷ $\ell_0$（乘法）。

**下標公式**：
- TODO 5：加法 `gamma * (yt - l - phi * b) + (1 - gamma) * s[t]`；乘法 `gamma * yt / (l + phi * b) + (1 - gamma) * s[t]`。
- TODO 6（`forecast`）：`trend_part = l + Phi_h * b`；`season = s[n + (h - 1) % m]`（`s` 的最後 $m$ 個就是最後一年的季節值）；加法相加、乘法相乘。

In [ ]:
from typing import Literal, Self

from scipy.optimize import minimize


class HoltWintersScratch:
    """SES／Holt／Holt-Winters（加法或乘法季節、可阻尼）。屬性名稱仿 statsmodels：fittedvalues_、params_、sse_。"""

    def __init__(self, trend: Literal["add"] | None = None, damped: bool = False, seasonal: Literal["add", "mul"] | None = None, m: int = M) -> None:
        self.trend, self.damped, self.seasonal, self.m = trend, damped, seasonal, m

    def _param_names(self) -> list[str]:
        return ["alpha"] + (["beta"] if self.trend else []) + (["gamma"] if self.seasonal else []) + (["phi"] if self.trend and self.damped else [])

    def _init_states(self, x: np.ndarray) -> tuple[float, float, np.ndarray | None]:
        m = self.m
        if self.seasonal is None:
            l0 = x[0]
            b0 = (x[1] - x[0]) if self.trend else 0.0
            return float(l0), float(b0), None
        l0 = x[:m].mean()
        b0 = (x[m : 2 * m].mean() - x[:m].mean()) / m if self.trend else 0.0
        s0 = x[:m] - l0 if self.seasonal == "add" else x[:m] / l0
        return float(l0), float(b0), s0

    def _filter(self, x: np.ndarray, p: dict, l0: float, b0: float, s0: np.ndarray | None) -> tuple[np.ndarray, float, float, list]:
        """跑一遍資料。回傳 (一步預測 (n,), 最後的 ℓ, 最後的 b, 季節清單 s（長度 n + m）)。"""
        alpha, beta, gamma = p["alpha"], p.get("beta", 0.0), p.get("gamma", 0.0)
        phi = p.get("phi", 1.0) if self.trend else 0.0  # 沒有趨勢時 b 一直是 0
        l, b = l0, (b0 if self.trend else 0.0)
        s = list(s0) if s0 is not None else [0.0 if self.seasonal != "mul" else 1.0] * self.m
        fitted = np.zeros(len(x))
        for t, yt in enumerate(x):
            st = s[t]
            fitted[t] = (l + phi * b) * st if self.seasonal == "mul" else l + phi * b + (st if self.seasonal == "add" else 0.0)
            deseason = yt - st if self.seasonal == "add" else (yt / st if self.seasonal == "mul" else yt)
            # TODO 3（程式）：水準 l_new
            raise NotImplementedError("TODO 3：水準")
            if self.trend:
                # TODO 4（程式）：趨勢 b_new
                raise NotImplementedError("TODO 4：趨勢")
            else:
                b_new = 0.0
            if self.seasonal:
                # TODO 5（程式）：季節 s_new（加法、乘法兩種；用的是舊的 l、b）
                raise NotImplementedError("TODO 5：季節")
            else:
                s_new = st
            s.append(s_new)
            l, b = l_new, b_new
        return fitted, l, b, s

    def fit(self, y: pd.Series, params: tuple | None = None, init: tuple | None = None) -> Self:
        """params=None：最小化一步誤差的 SSE 估計參數；否則用給定的 (alpha, beta, gamma, phi)（依有的元件）。
        init=None：用 _init_states；否則用給定的 (ℓ0, b0, s0)。"""
        self.y_ = y
        x = y.to_numpy(dtype=float)
        self.init_ = self._init_states(x) if init is None else init
        names = self._param_names()

        def sse(theta: np.ndarray) -> float:
            fitted = self._filter(x, dict(zip(names, theta)), *self.init_)[0]
            return float(((x - fitted) ** 2).sum())

        if params is None:
            x0 = [0.3] + ([0.1] if self.trend else []) + ([0.1] if self.seasonal else []) + ([0.95] if self.trend and self.damped else [])
            bounds = [(1e-4, 1 - 1e-4)] * len(x0)
            if self.trend and self.damped:
                bounds[-1] = (0.8, 0.995)
            res = minimize(sse, x0, method="L-BFGS-B", bounds=bounds)
            params = tuple(res.x)
        self.params_ = dict(zip(names, map(float, params)))
        self.fittedvalues_, self.level_, self.trend_, self.season_ = self._filter(x, self.params_, *self.init_)
        self.resid_ = x - self.fittedvalues_
        self.sse_ = float((self.resid_**2).sum())
        return self

    def forecast(self, h: int) -> np.ndarray:
        """h 步預測 (h,)。"""
        n, m = len(self.y_), self.m
        phi = self.params_.get("phi", 1.0)
        out = np.zeros(h)
        for k in range(1, h + 1):
            Phi_h = sum(phi**j for j in range(1, k + 1)) if self.trend else 0.0
            # TODO 6（程式）：第 k 步的預測（見 3C 表格最後一列；季節值 = self.season_[n + (k − 1) % m]）
            raise NotImplementedError("TODO 6：多步預測")
        return out

In [ ]:
# 最小例子：TODO 5、6（Holt-Winters 加法，m = 2）
# y = (12, 8, 14, 10)，m = 2；α = β = γ = 0.5、ℓ0 = 9、b0 = 1、季節初始值 (+2, −2)。先用迴圈算「正確答案」
y_hw = np.array([12.0, 8.0, 14.0, 10.0])
l_, b_, s_ = 9.0, 1.0, [2.0, -2.0]  # 讓第 1 期的預測 = 9 + 1 + 2 = 12（剛好對）
fit_hw = []
for t, yt in enumerate(y_hw):
    fit_hw.append(l_ + b_ + s_[t])
    l_new = 0.5 * (yt - s_[t]) + 0.5 * (l_ + b_)
    b_new = 0.5 * (l_new - l_) + 0.5 * b_
    s_.append(0.5 * (yt - l_ - b_) + 0.5 * s_[t])
    l_, b_ = l_new, b_new
print("一步預測：", fit_hw, "；最後 ℓ, b =", l_, b_, "；未來兩期：", [l_ + 1 * b_ + s_[4], l_ + 2 * b_ + s_[5]])
# 寫完 TODO 5、6 之後，取消下面的註解來對照（應跟上面的迴圈一樣：[12. 9. 13.25 10.0625] 與 [15.828125 11.84375]）：
# hw_mini = HoltWintersScratch(trend="add", seasonal="add", m=2).fit(pd.Series(y_hw), params=(0.5, 0.5, 0.5), init=(9.0, 1.0, np.array([2.0, -2.0]))); print(hw_mini.fittedvalues_, hw_mini.forecast(2))

### 3D. 估計參數、對照 statsmodels

**估計**：$\alpha, \beta, \gamma, (\phi)$ 讓**一步預測誤差的平方和** $\text{SSE} = \sum_t (y_t - \hat y_{t\mid t-1})^2$ 最小（`scipy.optimize.minimize`，範圍 0～1）。初始狀態固定用 `_init_states` 的值。

**對照**：statsmodels 的 `ExponentialSmoothing(..., initialization_method="known", initial_level=..., initial_trend=..., initial_seasonal=...)` 可以指定同樣的初始狀態：

1. **給定同樣的參數**（`optimized=False`）→ 一步預測應該**完全一樣**（差異 ~1e-12）；多步預測在 $h = m, 2m$（第 12、24 步）以外也完全一樣 —— statsmodels 預測第 $m$ 的倍數步時，用的是**上一輪**（$s_{n-m}$）而不是最新更新的季節值 $s_n$（實作細節；fpp3 的公式用 $s_n$，本章的手寫版照公式）；
2. **讓兩邊各自估計參數** → 最佳化演算法不同，參數可能略有差異，但 SSE 應該很接近。

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore", category=UserWarning)  # statsmodels 對 known 初始化的提醒
specs = {
    "SES": dict(trend=None, seasonal=None),
    "Holt": dict(trend="add", seasonal=None),
    "Holt damped": dict(trend="add", damped=True, seasonal=None),
    "HW additive": dict(trend="add", seasonal="add"),
    "HW multiplicative": dict(trend="add", seasonal="mul"),
    "HW mul damped": dict(trend="add", damped=True, seasonal="mul"),
}
check_rows, scratch_models = [], {}
for name, kw in specs.items():
    mine = HoltWintersScratch(**kw).fit(y_train)
    scratch_models[name] = mine
    l0, b0, s0 = mine.init_
    sm_kw = dict(trend=kw.get("trend"), damped_trend=kw.get("damped", False), seasonal=kw.get("seasonal"), seasonal_periods=M if kw.get("seasonal") else None,
                 initialization_method="known", initial_level=l0, initial_trend=b0 if kw.get("trend") else None, initial_seasonal=s0)
    p = mine.params_
    fixed = ExponentialSmoothing(y_train, **sm_kw).fit(smoothing_level=p["alpha"], smoothing_trend=p.get("beta"), smoothing_seasonal=p.get("gamma"),
                                                       damping_trend=p.get("phi"), optimized=False)
    est = ExponentialSmoothing(y_train, **sm_kw).fit()
    check_rows.append({"model": name, **{k: round(v, 4) for k, v in p.items()}, "SSE scratch": mine.sse_,
                       "max |fitted diff| (same params)": np.abs(fixed.fittedvalues.to_numpy() - mine.fittedvalues_).max(),
                       "max |forecast diff| h≠12,24": np.abs(np.delete(fixed.forecast(H).to_numpy() - mine.forecast(H), [M - 1, 2 * M - 1])).max(),
                       "|forecast diff| at h=12": abs(fixed.forecast(H).to_numpy()[M - 1] - mine.forecast(H)[M - 1]),
                       "SSE statsmodels (own estimate)": est.sse})
check = pd.DataFrame(check_rows).set_index("model")
display(check.round(6))

In [ ]:
# TODO 7（問答）
#
# (1) 給定同樣的參數時，一步預測的最大差異是多少？（應 < 1e-8）多步預測除了第 12、24 步以外呢？
#     第 12 步的差異只出現在有季節的模型 —— 用 3D 開頭的說明解釋：「同一個月」最新的季節值是哪一個？
# (2) 各自估計參數時，SSE 差多少？哪一邊比較小？為什麼可能不一樣？（提示：最佳化演算法、起始值、局部最小值）
# (3) 讀參數：
#     (a) SES 的 α 接近 1 還是 0？代表什麼？（提示：有很強的趨勢時，SES 只能「緊跟著最後一個值」）
#     (b) HW multiplicative 的 γ 是多少？γ 小代表季節變化很慢（季節形狀每年差不多）—— 跟 EDA 一致嗎？
# 答：

### 3E. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | $\alpha$ 的意義：權重 $\alpha(1-\alpha)^k$ 怎麼衰減？同一條序列用不同的 $\alpha$，平滑的結果長怎樣？ |
| 2 | 六個模型在 test 上的預測：誰抓到了趨勢？誰抓到了季節變大？ |
| 3 | 乘法季節 vs「取 log 再用加法季節」 |

In [ ]:
# 實驗 1：α 的權重與平滑效果
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
for a in [0.1, 0.3, 0.7]:
    axes[0].plot(np.arange(20), a * (1 - a) ** np.arange(20), "o-", ms=3, label=f"α = {a}")
    ses_a = HoltWintersScratch().fit(y_train, params=(a,))
    axes[1].plot(y_train.index, ses_a.fittedvalues_, label=f"SES α = {a}")
axes[0].set_xlabel("k (months ago)")
axes[0].set_ylabel("weight α(1−α)^k")
axes[0].legend()
axes[1].plot(y_train.index, y_train.to_numpy(), color="k", lw=1, label="actual")
axes[1].legend()
axes[1].set_title("one-step-ahead fitted values")
plt.show()
for a in [0.1, 0.3, 0.7]:
    print(f"α = {a}：權重的「半衰期」= {np.log(0.5) / np.log(1 - a):.1f} 個月；過去 12 個月的權重總和 = {1 - (1 - a) ** 12:.3f}")

In [ ]:
# 實驗 2：六個模型在 test 上的預測（只看圖與 MASE；選模型在第 5 節用 CV）
fig, ax = plt.subplots(figsize=(14, 4.5))
y.iloc[-48:].plot(ax=ax, color="k", lw=2, label="actual")
exp2 = {}
for name, mdl in scratch_models.items():
    fc = pd.Series(mdl.forecast(H), index=y_test.index)
    exp2[name] = forecast_metrics(y_test, fc, y_train, season=M)
    fc.plot(ax=ax, ls="--", label=name)
ax.legend(fontsize=8)
plt.show()
display(pd.DataFrame(exp2).T[["mae", "rmse", "smape", "mase"]])

In [ ]:
# 實驗 3：乘法季節 vs log + 加法季節（預測再 exp 回來）
log_hw = HoltWintersScratch(trend="add", seasonal="add").fit(np.log(y_train))
fc_log = np.exp(log_hw.forecast(H))
fc_mul = scratch_models["HW multiplicative"].forecast(H)
print("log + 加法 的參數：", {k: round(v, 3) for k, v in log_hw.params_.items()})
display(pd.DataFrame({"HW multiplicative": forecast_metrics(y_test, fc_mul, y_train, season=M),
                      "log + HW additive, exp back": forecast_metrics(y_test, fc_log, y_train, season=M)}).T[["mae", "rmse", "smape", "mase"]])

In [ ]:
# TODO 8（問答）實驗 1–3
#
# (1) 實驗 1：α = 0.1、0.3、0.7 的半衰期各是幾個月？α 大的 SES 曲線比較「跟得緊」還是比較「平滑」？
#     SES 的一步預測為什麼總是落後實際值（尤其在上升的地方）？
# 答：
#
# (2) 實驗 2：
#     (a) SES、Holt 的預測是什麼形狀？為什麼它們抓不到季節？
#     (b) HW additive 和 HW multiplicative 在 test 的最後一年差最多 —— 誰的季節波動比較大？哪一個比較接近實際？跟 TODO 1 (2)(b) 的判斷一致嗎？
#     (c) 阻尼趨勢讓預測變好還是變差？在這份資料上（穩定成長的航空業）阻尼合理嗎？
# 答：
#
# (3) 實驗 3：log + 加法 跟 乘法 誰比較準？兩者的差別在哪裡？（提示：乘法模型的「趨勢」是加法的 —— 每年多固定人數；log 模型的趨勢在 log 尺度上是加法 → 原尺度是「每年成長固定比例」）
# 答：

---
## 4. 套件實作：statsmodels `ExponentialSmoothing`／`ETSModel`

| 類別 | 用途 |
| --- | --- |
| `ExponentialSmoothing(y, trend, damped_trend, seasonal, seasonal_periods, initialization_method)` | 3D 已經用過：經典 Holt-Winters，點預測 |
| `ETSModel(y, error="add"／"mul", trend, seasonal, ...)` | 狀態空間版本（Hyndman et al. 2002）：同樣的遞迴，但多了**誤差的機率模型** → 可以用最大概似估計參數（含初始狀態）、算 AIC、給**預測區間** |

ETS 的命名：(Error, Trend, Seasonal)，例如 ETS(M, A, M) = 乘法誤差、加法趨勢、乘法季節；ETS(A, Ad, A) = 加法誤差、阻尼趨勢、加法季節。

下面用 `ETSModel` 估計三個常見的組合，比較 AIC，並畫出 80%／95% 預測區間。

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

ets_specs = {
    "ETS(A,A,A)": dict(error="add", trend="add", seasonal="add"),
    "ETS(M,A,M)": dict(error="mul", trend="add", seasonal="mul"),
    "ETS(M,Ad,M)": dict(error="mul", trend="add", damped_trend=True, seasonal="mul"),
}
ets_rows, ets_fits = [], {}
for name, kw in ets_specs.items():
    res = ETSModel(y_train, seasonal_periods=M, **kw).fit(disp=False)
    ets_fits[name] = res
    fc = res.forecast(H)
    ets_rows.append({"model": name, "AIC": res.aic, "BIC": res.bic, **forecast_metrics(y_test, fc, y_train, season=M)})
display(pd.DataFrame(ets_rows).set_index("model")[["AIC", "BIC", "mae", "rmse", "mase"]].round(3))

best_ets = min(ets_fits, key=lambda k: ets_fits[k].aic)
pred = ets_fits[best_ets].get_prediction(start=len(y_train), end=len(y_train) + H - 1)
sf = pred.summary_frame(alpha=0.05)
sf80 = pred.summary_frame(alpha=0.2)
fig, ax = plt.subplots(figsize=(14, 4))
y.iloc[-60:].plot(ax=ax, color="k", label="actual")
ax.plot(y_test.index, sf["mean"], color="tab:red", label=f"{best_ets} (lowest AIC)")
ax.fill_between(y_test.index, sf["pi_lower"], sf["pi_upper"], alpha=0.15, color="tab:red", label="95% PI")
ax.fill_between(y_test.index, sf80["pi_lower"], sf80["pi_upper"], alpha=0.3, color="tab:red", label="80% PI")
ax.legend()
plt.show()
inside = ((y_test.to_numpy() >= sf["pi_lower"].to_numpy()) & (y_test.to_numpy() <= sf["pi_upper"].to_numpy())).mean()
print(f"{best_ets}：test 的 24 個月有 {inside:.0%} 落在 95% 預測區間內")

In [ ]:
# TODO 9（問答）ETS
#
# (1) 三個 ETS 模型的 AIC 誰最低？AIC 最低的，test 的 MASE 也最低嗎？（AIC 只用訓練資料，衡量「一步」的配適；test 是 1～24 步）
# (2) 預測區間：越往後越寬嗎？為什麼？95% 區間實際涵蓋了 test 的幾 %？
# (3) ExponentialSmoothing 和 ETSModel 的點預測公式一樣，差在哪裡？（提示：初始狀態怎麼估、有沒有誤差分布）
# 答：

---
## 5. 調參：滾動起點的交叉驗證（rolling-origin CV）

時間序列不能用一般的 k-fold（會用未來預測過去）。**滾動起點**：

```
fold 1: train = y[:84]  → 預測 y[84:96]
fold 2: train = y[:96]  → 預測 y[96:108]
fold 3: train = y[:108] → 預測 y[108:120]
```

每個 fold 都只用「起點之前」的資料估計，預測接下來 12 個月；三個 fold 的 MASE 平均 → 選模型。全程只用 `y_train`（120 個月），test 不碰。

**候選**：6 個手寫模型 + 「log + HW additive」（實驗 3）+ 季節性 naive。

In [ ]:
def rolling_origin_cv(y: pd.Series, make_forecast, origins: list[int], horizon: int = 12, m: int = M) -> pd.DataFrame:
    """make_forecast(train_series, horizon) → (horizon,) 的預測。回傳每個 fold 的 forecast_metrics。"""
    rows = []
    for origin in origins:
        # TODO 10（程式）：
        #   - tr = y.iloc[:origin]；va = y.iloc[origin : origin + horizon]
        #   - fc = make_forecast(tr, horizon)
        #   - rows.append({"origin": str(y.index[origin].date()), **forecast_metrics(va, fc, tr, season=m)})
        #   常見錯誤：用整條 y 當 forecast_metrics 的 y_train（MASE 的分母只能用「這個 fold 的訓練資料」）
        raise NotImplementedError("TODO 10：滾動起點")
    return pd.DataFrame(rows).set_index("origin")


ORIGINS = [84, 96, 108]
cv_rows = {}
candidates = {"snaive": lambda tr, h: baseline_forecast(tr, h, "snaive").to_numpy()}
candidates.update({name: (lambda kw: (lambda tr, h: HoltWintersScratch(**kw).fit(tr).forecast(h)))(kw) for name, kw in specs.items()})
candidates["log + HW additive"] = lambda tr, h: np.exp(HoltWintersScratch(trend="add", seasonal="add").fit(np.log(tr)).forecast(h))
for name, f in candidates.items():
    cv = rolling_origin_cv(y_train, f, ORIGINS)
    cv_rows[name] = {"mean MASE": cv["mase"].mean(), "std MASE": cv["mase"].std(), **{f"MASE @{o}": v for o, v in cv["mase"].items()}}
cv_table = pd.DataFrame(cv_rows).T.sort_values("mean MASE")
display(cv_table.round(3))

In [ ]:
# 選模型：mean MASE 最低；差距在一個「fold 間標準差 / √3」以內時選比較簡單的（參數比較少）
N_PARAMS = {"snaive": 0, "SES": 1, "Holt": 2, "Holt damped": 3, "HW additive": 3, "HW multiplicative": 3, "log + HW additive": 3, "HW mul damped": 4}
best = cv_table.index[0]
se_best = cv_table.loc[best, "std MASE"] / np.sqrt(len(ORIGINS))
close = cv_table[cv_table["mean MASE"] <= cv_table.loc[best, "mean MASE"] + se_best].index.tolist()
chosen = min(close, key=lambda k: (N_PARAMS[k], cv_table.loc[k, "mean MASE"]))
print(f"CV 最好：{best}（mean MASE {cv_table.loc[best, 'mean MASE']:.3f} ± SE {se_best:.3f}）；一個標準誤內：{close} → 選 {chosen}")

In [ ]:
# TODO 11（問答）調參
#
# (1) 三個 fold 的 MASE：同一個模型在不同 fold 差很多嗎？哪個 fold 最難？為什麼？
# (2) CV 選出的模型，跟實驗 2 在 test 上看起來最好的一樣嗎？為什麼「在 test 上挑」是作弊？
# (3) 只有 3 個 fold、每個 fold 12 個點，CV 的結論可靠嗎？你可以怎麼讓它更可靠？（提示：更多起點、每次只往前推 1 個月）
# 答：

---
## 6. 測試集評估

用第 5 節選出的模型，在完整的 `y_train`（120 個月）上估計參數，預測 test 的 24 個月，只評估一次。也記下季節性 naive 與 ETS（AIC 最低）當參照點。成績寫進 `results/air_passengers.csv`（下一章 ARIMA 會接著加）。

In [ ]:
from learning_map.results import compare, log_result

BENCH = "air_passengers"
if chosen == "snaive":
    final, fc_final = None, base_fc["snaive"].to_numpy()
elif chosen == "log + HW additive":
    final = HoltWintersScratch(trend="add", seasonal="add").fit(np.log(y_train))
    fc_final = np.exp(final.forecast(H))
else:
    final = HoltWintersScratch(**specs[chosen]).fit(y_train)
    fc_final = final.forecast(H)
eval_models = {
    "Seasonal naive": (base_fc["snaive"].to_numpy(), {}),
    f"{chosen} (scratch, CV-selected)": (fc_final, {k: round(v, 4) for k, v in (final.params_ if final else {}).items()}),
    f"{best_ets} (statsmodels, lowest AIC)": (ets_fits[best_ets].forecast(H).to_numpy(), {"aic": round(float(ets_fits[best_ets].aic), 2)}),
}
test_scores = {}
for name, (fc, params) in eval_models.items():
    m_ = forecast_metrics(y_test, fc, y_train, season=M)
    test_scores[name] = m_
    log_result(BENCH, name, m_, horizon=H, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "mase", higher_is_better=False)

---
## 7. 診斷

1. **殘差**（一步預測誤差）：應該像白雜訊 —— 平均 0、沒有自我相關、變異固定。用 ACF 與 Ljung-Box 檢定（虛無假設：前 $k$ 個 lag 的自我相關都是 0）。
2. **誤差隨預測步數**：test 上第 1～24 步的絕對誤差。

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

diag_model = final if final is not None else scratch_models["HW multiplicative"]
print("診斷的模型：", chosen if final is not None else "HW multiplicative", "（log 模型的殘差是 log 尺度：0.05 ≈ 5%）" if chosen == "log + HW additive" else "")
resid = pd.Series(diag_model.resid_[M:], index=y_train.index[M:])  # 去掉第一年（初始化期）
fig, axes = plt.subplots(1, 3, figsize=(18, 3.5))
resid.plot(ax=axes[0])
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_title("one-step residuals (train)")
r = acf(resid.to_numpy(), 24)
axes[1].bar(np.arange(25), r)
axes[1].axhline(2 / np.sqrt(len(resid)), color="gray", ls=":")
axes[1].axhline(-2 / np.sqrt(len(resid)), color="gray", ls=":")
axes[1].set_title("residual ACF")
err = np.abs(y_test.to_numpy() - fc_final)
axes[2].bar(np.arange(1, H + 1), err)
axes[2].set_xlabel("horizon h")
axes[2].set_title("|test error| by horizon")
plt.show()
display(acorr_ljungbox(resid, lags=[6, 12, 24]).round(4))
print("殘差平均：", round(resid.mean(), 3), "；前半 std：", round(resid.iloc[: len(resid) // 2].std(), 2), "；後半 std：", round(resid.iloc[len(resid) // 2 :].std(), 2))

In [ ]:
# TODO 12（問答，本章總結）
#
# (1) 測試集：選出的模型 MASE 是多少？比季節性 naive 好多少？跟 ETS（AIC 最低）比呢？
# (2) 殘差：
#     (a) Ljung-Box 的 p 值在 lag 12、24 是多少？殘差裡還有沒有被模型漏掉的結構？
#     (b) 前半段與後半段的殘差 std 一樣嗎？如果後半段比較大，代表什麼（提示：乘法 vs 加法、log）？
# (3) 誤差隨步數：第 13～24 步的誤差比第 1～12 步大嗎？為什麼？
# (4) 下一章 ARIMA：指數平滑是「把序列拆成水準、趨勢、季節，分別平滑」；ARIMA 是「把序列差分到平穩，再用過去的值與過去的誤差回歸」。你猜哪一個在這份資料上比較準？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

AirPassengers（train 1949–1958、test 1959–1960，24 步預測）

| 方法 | 做法 | 參數 | CV mean MASE | test MASE | test RMSE |
| --- | --- | --- | --- | --- | --- |
| 季節性 naive | | | | | |
| SES | | | | | |
| Holt（含阻尼） | | | | | |
| HW additive | | | | | |
| HW multiplicative（含阻尼） | | | | | |
| ETS（AIC 最低） | | | — | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ ETS 的狀態空間形式與最大概似估計（Hyndman et al. 2008, *Forecasting with Exponential Smoothing*）
- ☐ 預測區間的推導（加法誤差模型的變異數公式；乘法模型用模擬）
- ☐ 指數平滑與 ARIMA 的等價關係（SES = ARIMA(0,1,1)；Holt = ARIMA(0,2,2)）

**延伸**
- ☐ 自動選模型（`statsforecast` 的 AutoETS）
- ☐ 多重季節（日資料的週 + 年季節）：TBATS、MSTL（第 6 章 STL）
- ☐ 滾動起點 CV 每次只推 1 個月、多個 horizon 分開評估